# Semana 9: RQ4, camadas auxiliares do Earth Engine (água permanente e uso do solo)

Seção 2 de `prereg/plano_rq4.md` (entradas de 2026-11-02). **Este notebook não roda nenhum modelo e não lê nenhuma referência de enchente.** Ele pede ao Google Earth Engine duas camadas para os três tiles (22JDM, 22JCN, 22JDN), na mesma grade de 10 m do Sentinel-2 já exportado:

| Arquivo | Coleção | Uso |
|---|---|---|
| `..._gsw.tif` | `JRC/GSW1_4/GlobalSurfaceWater`, banda `occurrence` (0 a 100), uint8, sem dado = 0 | Água permanente: ocorrência ≥ 90%, que entra na água de cada referência (plano, seção 2). A camada original é de 30 m e entra por vizinho mais próximo |
| `..._worldcover.tif` | `ESA/WorldCover/v200` (mapa de 2021), banda `Map`, uint8, sem dado = 0 | Estratos da seção 3: área construída (classe 50), água permanente (80) e o resto |

Como na exportação do Sentinel-2, o Earth Engine pode cortar o arquivo; a célula 4 o coloca na grade inteira de 10980 × 10980 de cada tile por deslocamento de pixels inteiros, sem reamostrar, monta os arquivos no disco local, copia para o Drive e confere os SHA-256 depois de desmontar e montar o Drive de novo. Um arquivo que já está no Drive nunca é regravado: se ele já existe, a célula só confere que o conteúdo é o mesmo que sai do arquivo bruto.

| Célula | O que faz |
|---|---|
| 1 | Monta o Drive, clona o repositório, entra no Earth Engine, lê a grade de cada tile do registro do Sentinel-2 |
| 2 | Descreve as duas camadas (versão, bandas) |
| 3 | Pede as seis exportações (2 por tile) e espera terminar (poucos minutos) |
| 4 | Coloca cada arquivo na grade inteira, confere, copia para o Drive, remonta o Drive e confere de novo, grava `rq4_auxiliares.json` |

Abra o Colab logada na conta do mestrado (não precisa de GPU). Se alguma célula der erro, pare e me mande a saída inteira. A saída de cada rodada vai para uma pasta nova (`rq4_auxiliares`, depois `rq4_auxiliares_2`, e assim por diante).

Depois da célula 4, me mande `rq4_auxiliares.json` e `rq4_auxiliares_sha256.csv` da pasta de saída desta rodada.

In [ ]:
# Célula 1 de 4: Drive, repositório e Earth Engine
import os, re, json, time, hashlib, glob, shutil, subprocess, sys, csv
import numpy as np

from google.colab import drive
drive.mount('/content/drive')
MEUDRIVE = '/content/drive/MyDrive'
RAIZ = f'{MEUDRIVE}/Mestrado'
assert os.path.isdir(f'{RAIZ}/modelo_fixado'), 'Pasta Mestrado/modelo_fixado não encontrada. Confira se o Colab está na conta do mestrado.'

DESTINO = f'{RAIZ}/dados/rs2024/aux'
PASTA_EXPORT = 'rq4_aux'                     # pasta criada pelo Earth Engine na raiz do seu Drive
SAIDA_BASE = f'{RAIZ}/resultados/semana9/rq4_auxiliares'
SAIDA_DRIVE, n = SAIDA_BASE, 1
while os.path.isdir(SAIDA_DRIVE) and os.listdir(SAIDA_DRIVE):      # nunca sobrescreve
    n += 1
    SAIDA_DRIVE = f'{SAIDA_BASE}_{n}'
print('Saída desta rodada:', SAIDA_DRIVE.replace(MEUDRIVE, 'MyDrive'))
os.makedirs(DESTINO, exist_ok=True)

REPO_URL = 'https://github.com/RoseBorges44/prithvi-confidence-shift'
REPO_GIT = '/content/repo'
if not os.path.isdir(REPO_GIT):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO_GIT], check=True)
else:
    subprocess.run(['git', '-C', REPO_GIT, 'pull', '-q'], check=True)
commit = subprocess.run(['git', '-C', REPO_GIT, 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
sys.path.insert(0, f'{REPO_GIT}/analysis/rq4')
import s2_exportacao as S
SHA_MODULO = hashlib.sha256(open(f'{REPO_GIT}/analysis/rq4/s2_exportacao.py', 'rb').read()).hexdigest()

def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with open(caminho, 'rb') as f:
        for bloco in iter(lambda: f.read(1 << 24), b''):
            h.update(bloco)
    return h.hexdigest()

# A grade de cada tile é a do Sentinel-2 já exportado (registro do repositório).
REGISTRO = json.load(open(f'{REPO_GIT}/analysis/rq4/entradas/rq4_entradas.json'))
GRADES = {}
for e in REGISTRO['entradas']:
    g = e['grade']
    assert g['crs'] == S.CRS_ESPERADO[e['tile']] and (g['altura'], g['largura']) == (S.LADO, S.LADO)
    GRADES[e['tile']] = {'crs': g['crs'], 'transform': g['transform'], 'origem': (g['transform'][2], g['transform'][5])}
assert set(GRADES) == set(S.TILES), sorted(GRADES)

import ee
PROJETO_GEE = 'mestrado-prithvi'              # ID do projeto do Earth Engine (já preenchido)
try:
    ee.Initialize(project=PROJETO_GEE)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJETO_GEE)
print(f'Repositório no commit {commit[:7]}. Earth Engine pronto (projeto {PROJETO_GEE}). Grades:', {t: g['origem'] for t, g in GRADES.items()})

In [ ]:
# Célula 2 de 4: descrição das camadas
GSW_ASSET = 'JRC/GSW1_4/GlobalSurfaceWater'
WC_ASSET = 'ESA/WorldCover/v200'
gsw = ee.Image(GSW_ASSET)
wc = ee.ImageCollection(WC_ASSET).first()
info_gsw, info_wc = gsw.getInfo(), wc.getInfo()
print('GSW:', info_gsw['id'], '| bandas', [b['id'] for b in info_gsw['bands']], '| versão', info_gsw['properties'].get('system:version', ''))
print('WorldCover:', info_wc['id'], '| bandas', [b['id'] for b in info_wc['bands']])
ASSETS = {'gsw': {'asset': GSW_ASSET, 'banda': 'occurrence', 'id': info_gsw['id'], 'propriedades': {k: v for k, v in info_gsw['properties'].items() if k.startswith(('system:version', 'system:time'))}},
          'worldcover': {'asset': WC_ASSET, 'banda': 'Map', 'id': info_wc['id'], 'propriedades': {k: v for k, v in info_wc['properties'].items() if k.startswith(('system:version', 'system:time'))}}}
assert info_wc['id'].endswith('2021'), 'O mapa do WorldCover não é o de 2021. Pare e me mande esta saída.'

def imagens():
    return {'gsw': gsw.select('occurrence').toUint8().unmask(0),
            'worldcover': wc.select('Map').toUint8().unmask(0)}

def regiao(tile):
    x0, y0 = GRADES[tile]['origem']
    n = S.LADO * S.PIXEL
    return ee.Geometry.Rectangle([x0, y0 - n, x0 + n, y0], proj=GRADES[tile]['crs'], geodesic=False)

In [ ]:
# Célula 3 de 4: exportações
ATIVAS = {t.config.get('description'): t for t in ee.batch.Task.list() if t.state in ('READY', 'RUNNING')}
TAREFAS = {}
for tile in S.TILES:
    for nome, img in imagens().items():
        prefixo = f'{tile}_{nome}'
        ja = glob.glob(f'{MEUDRIVE}/{PASTA_EXPORT}/{prefixo}*.tif') + glob.glob(f'{DESTINO}/{prefixo}*.tif')
        if ja:
            print(f'{prefixo}: já exportado ({len(ja)} arquivo).')
            continue
        if prefixo in ATIVAS:
            TAREFAS[prefixo] = ATIVAS[prefixo]
            print(f'{prefixo}: tarefa já em andamento, não repito.')
            continue
        tarefa = ee.batch.Export.image.toDrive(
            image=img, description=prefixo, folder=PASTA_EXPORT, fileNamePrefix=prefixo,
            region=regiao(tile), crs=GRADES[tile]['crs'], crsTransform=GRADES[tile]['transform'],
            maxPixels=int(1e10), fileFormat='GeoTIFF')
        tarefa.start()
        TAREFAS[prefixo] = tarefa
        print(f'{prefixo}: tarefa pedida.')

limite = time.time() + 2 * 3600
ultimo = None
while TAREFAS and time.time() < limite:
    estados = {p: t.status()['state'] for p, t in TAREFAS.items()}
    if estados != ultimo:
        print(time.strftime('%H:%M:%S'), estados)
        ultimo = estados
    ruins = {p: s for p, s in estados.items() if s in ('FAILED', 'CANCELLED', 'CANCEL_REQUESTED')}
    assert not ruins, f'Exportação falhou: {ruins}. Pare e me mande esta saída.'
    if all(s == 'COMPLETED' for s in estados.values()):
        break
    time.sleep(30)
else:
    if TAREFAS:
        raise RuntimeError('As exportações não terminaram em 2 horas. Rode esta célula de novo mais tarde.')
print('Exportações concluídas.')

In [ ]:
# Célula 4 de 4: grade inteira, conferência e registro
import rasterio

def esperar(caminho, segundos=180):
    # O Drive montado no Colab às vezes demora a mostrar um arquivo recém-gravado ou movido.
    os.sync()
    for _ in range(segundos // 5):
        if os.path.exists(caminho):
            return caminho
        time.sleep(5)
    pasta = os.path.dirname(caminho)
    raise FileNotFoundError(f'{caminho} não apareceu em {segundos} s. Pasta: {sorted(os.listdir(pasta))[:40] if os.path.isdir(pasta) else "não existe"}')

CANON = f'{DESTINO}/canonico'
LOCAL = '/content/rq4_aux_canonico'
os.makedirs(CANON, exist_ok=True)
os.makedirs(LOCAL, exist_ok=True)
ARQUIVOS, LINHAS, ESTATISTICAS = {}, [], {}
for tile in S.TILES:
    for nome in ('gsw', 'worldcover'):
        prefixo = f'{tile}_{nome}'
        origem = glob.glob(f'{MEUDRIVE}/{PASTA_EXPORT}/{prefixo}*.tif')
        destino = glob.glob(f'{DESTINO}/{prefixo}*.tif')
        achados = destino or origem
        assert len(achados) == 1, f'{prefixo}: {len(achados)} arquivos (o esperado é 1). Pare e me mande esta saída.'
        bruto = achados[0]
        if not destino:
            novo = f'{DESTINO}/{os.path.basename(bruto)}'
            shutil.move(bruto, novo)
            bruto = esperar(novo)
        canon_drive = f'{CANON}/{prefixo}.tif'
        canon = f'{LOCAL}/{prefixo}.tif'
        sha_bruto = sha256_arquivo(bruto)
        with rasterio.open(bruto) as s:
            perfil = {'height': s.height, 'width': s.width, 'count': s.count, 'dtype': s.dtypes[0],
                      'crs': s.crs.to_string(), 'transform': tuple(s.transform)[:6]}
            col_off, row_off = S.deslocamento_na_grade(nome, perfil, GRADES[tile]['origem'], tile)
            x0, y0 = GRADES[tile]['origem']
            novo_perfil = {'driver': 'GTiff', 'height': S.LADO, 'width': S.LADO, 'count': 1, 'dtype': 'uint8', 'crs': s.crs,
                           'transform': rasterio.Affine(10.0, 0.0, x0, 0.0, -10.0, y0),
                           'compress': 'deflate', 'predictor': 2, 'tiled': True, 'blockxsize': 512, 'blockysize': 512}
            esperado = S.colocar_na_grade(s.read(1)[None], col_off, row_off, 0)[0]      # o que a grade inteira deve conter
        existe = os.path.exists(canon_drive)
        if existe:
            shutil.copyfile(canon_drive, canon)
            with rasterio.open(canon) as s:
                assert (s.height, s.width, s.count, s.dtypes[0]) == (S.LADO, S.LADO, 1, 'uint8') and tuple(s.transform)[:6] == (10.0, 0.0, x0, 0.0, -10.0, y0), prefixo
                assert np.array_equal(s.read(1), esperado), (
                    f'{prefixo}: o arquivo que já está no Drive não tem o conteúdo que sai do bruto. Não regravei nada. Pare e me mande esta saída.')
        else:
            with rasterio.open(canon + '.parcial.tif', 'w', **novo_perfil) as o:
                o.write(esperado, 1)
            os.replace(canon + '.parcial.tif', canon)
            shutil.copyfile(canon, canon_drive)
            esperar(canon_drive)
        sha_c = sha256_arquivo(canon)
        assert sha256_arquivo(canon_drive) == sha_c, f'{prefixo}: a cópia no Drive difere do arquivo local. Pare e me mande esta saída.'
        if nome == 'gsw':
            ESTATISTICAS[prefixo] = {'fracao_ocorrencia_maior_igual_90': float((esperado >= 90).mean()), 'fracao_zero': float((esperado == 0).mean()),
                                     'maximo': int(esperado.max())}
            assert esperado.max() <= 100, f'{prefixo}: ocorrência acima de 100. Pare e me mande esta saída.'
        else:
            valores, cont = np.unique(esperado, return_counts=True)
            ESTATISTICAS[prefixo] = {'classes': {int(v): int(c) for v, c in zip(valores, cont)}, 'fracao_sem_dado': float((esperado == 0).mean())}
        print(f'{prefixo}: arquivo do Earth Engine {perfil["height"]} x {perfil["width"]}, deslocamento ({col_off}, {row_off}) na grade; {"já existia no Drive, conferido" if existe else "gravado"}; {ESTATISTICAS[prefixo]}')
        ARQUIVOS[prefixo] = {'bruto': {'arquivo': os.path.basename(bruto), 'bytes': os.path.getsize(bruto), 'sha256': sha_bruto,
                                       'altura': perfil['height'], 'largura': perfil['width'], 'transform': list(perfil['transform'])},
                             'canonico': {'arquivo': f'canonico/{prefixo}.tif', 'bytes': os.path.getsize(canon), 'sha256': sha_c},
                             'deslocamento_pixels': {'coluna': col_off, 'linha': row_off}, 'estatisticas': ESTATISTICAS[prefixo]}
        LINHAS.append([os.path.basename(bruto), 'bruto', ARQUIVOS[prefixo]['bruto']['bytes'], sha_bruto])
        LINHAS.append([f'canonico/{prefixo}.tif', 'canonico', ARQUIVOS[prefixo]['canonico']['bytes'], sha_c])
        del esperado

# Conferência final contra o que o Drive guardou de verdade: desmonta (grava tudo), monta de novo e relê os SHA-256.
print('Gravando o Drive e conferindo de novo; pode levar alguns minutos.')
drive.flush_and_unmount()
time.sleep(10)
drive.mount('/content/drive', force_remount=True)
for nome_arq, tipo, _, sha in LINHAS:
    caminho = f'{DESTINO}/{nome_arq}'
    assert os.path.exists(caminho) and sha256_arquivo(caminho) == sha, f'{nome_arq}: o Drive não guardou o arquivo como esperado. Pare e me mande esta saída.'
print('Todos os arquivos do Drive conferem depois de remontar.')

os.makedirs(SAIDA_DRIVE, exist_ok=True)
INFO = {'plano': 'prereg/plano_rq4.md, secao 2', 'commit_repositorio': commit, 's2_exportacao_sha256': SHA_MODULO,
        'camadas': ASSETS, 'grades': {t: {'crs': g['crs'], 'transform': g['transform']} for t, g in GRADES.items()},
        'arquivos': ARQUIVOS, 'gerado_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
json.dump(INFO, open(f'{SAIDA_DRIVE}/rq4_auxiliares.json', 'w'), indent=2, ensure_ascii=False)
with open(f'{SAIDA_DRIVE}/rq4_auxiliares_sha256.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['arquivo', 'tipo', 'bytes', 'sha256']); w.writerows(LINHAS)
print(open(f'{SAIDA_DRIVE}/rq4_auxiliares_sha256.csv').read())
print(f'Registro em {SAIDA_DRIVE}. Nenhum modelo foi rodado.')